# NB_Silver_Programmes
**PNG Masalais (fictional club, synthetic data)**. Field app files (Bronze `Files/field`) to Silver tables.

Run every cell, top to bottom. If you need to redo anything, use **Run all** so the DQ log isn't doubled.

Principles:
1. **Correct what can be corrected** (gender spellings, province derivable from the school), and log it.
2. **Remove only what isn't real** (the same session submitted twice).
3. **Quarantine what can't be attributed** (attendance for a participant who isn't in the register).
   Those rows can't be counted by gender or age, so they would distort reach. They're excluded and logged,
   so programme officers can chase the missing registrations.

Default lakehouse: **LH_Bronze**. Writes go to **LH_Silver**. Both are without schemas.

## 1. Setup

In [ ]:
from pyspark.sql import functions as F, Window
from datetime import datetime, timezone
from functools import reduce

BRONZE = "LH_Bronze"
SILVER = "LH_Silver"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
FIELD = "Files/field"

def read_csv(path):
    # Everything arrives as text; we set types deliberately below.
    return spark.read.option("header", True).csv(path)

dq_frames = []
def log_issues(df, source_table, rule, action, key_col, detail_expr):
    dq_frames.append(df.select(
        F.lit(RUN_ID).alias("RunId"), F.lit(source_table).alias("SourceTable"),
        F.lit(rule).alias("Rule"), F.lit(action).alias("Action"),
        F.col(key_col).cast("string").alias("RecordKey"), detail_expr.cast("string").alias("Detail")))
print("Run", RUN_ID)

## 2. Reference: provinces and schools

In [ ]:
provinces = read_csv("Files/reference/Provinces.csv")
schools = (read_csv(f"{FIELD}/schools.csv")
           .withColumn("ProgrammeJoinDate", F.to_date("ProgrammeJoinDate")))
print("Provinces:", provinces.count(), "| Schools:", schools.count())

## 3. Participant register: standardise gender, add age band
The app let officers type gender freely, so we have `F`, `Female`, `female`, `f`, ` F` and so on.
Rule: trim, uppercase, take the first letter. Anything that isn't then `F` or `M` is flagged, not guessed.

Age band is worked out from birth year at registration. Bands match how sport development programmes usually report.

In [ ]:
p = read_csv(f"{FIELD}/participants_register.csv")
p = (p.withColumn("GenderRaw", F.col("Gender"))
      .withColumn("Gender", F.upper(F.substring(F.trim("Gender"), 1, 1)))
      .withColumn("BirthYear", F.col("BirthYear").cast("int"))
      .withColumn("RegisteredDate", F.to_date("RegisteredDate")))

fixed = p.filter(F.col("GenderRaw") != F.col("Gender"))
log_issues(fixed, "participants_register", "Non-standard gender value", "Corrected", "ParticipantID",
           F.concat(F.lit("'"), F.col("GenderRaw"), F.lit("' -> "), F.col("Gender")))
unknown_g = p.filter(~F.col("Gender").isin("F", "M"))
log_issues(unknown_g, "participants_register", "Gender not recognised", "Flagged", "ParticipantID", F.col("GenderRaw"))

age = F.year("RegisteredDate") - F.col("BirthYear")
p = (p.withColumn("AgeAtRegistration", age)
      .withColumn("AgeBand", F.when(age < 12, "Under 12").when(age <= 15, "12-15").when(age <= 18, "16-18")
                              .when(age <= 24, "19-24").otherwise("25+"))
      .withColumn("GenderLabel", F.when(F.col("Gender") == "F", "Female")
                                  .when(F.col("Gender") == "M", "Male").otherwise("Unknown"))
      .drop("GenderRaw"))
print("Participants:", p.count(), "| gender corrected:", fixed.count(), "| unrecognised:", unknown_g.count())

## 4. Sessions: remove resubmissions, fill missing provinces
- Officers sometimes edit a session and submit again. Same `SessionID`, later `SubmittedAt`.
  The **latest** submission is the corrected one, so we keep that and log the earlier copy.
- School sessions sometimes have a blank province. A school never moves, so we take the province from the school.

In [ ]:
s = (read_csv(f"{FIELD}/sessions/*.csv")
     .withColumn("SessionDate", F.to_date("SessionDate"))
     .withColumn("DurationMins", F.col("DurationMins").cast("int"))
     .withColumn("SubmittedAt", F.to_timestamp("SubmittedAt"))
     .withColumn("SchoolID", F.nullif(F.trim("SchoolID"), F.lit("")))
     .withColumn("ProvinceCode", F.nullif(F.trim("ProvinceCode"), F.lit(""))))
raw_count = s.count()

w = Window.partitionBy("SessionID").orderBy(F.col("SubmittedAt").desc())
s = s.withColumn("_rn", F.row_number().over(w))
resub = s.filter("_rn > 1")
log_issues(resub, "sessions", "Duplicate submission (kept latest)", "Removed", "SessionID",
           F.concat_ws(" | ", F.col("SubmittedAt").cast("string"), F.col("DurationMins").cast("string")))
s = s.filter("_rn = 1").drop("_rn")

s = (s.join(schools.select("SchoolID", F.col("ProvinceCode").alias("SchoolProvince")), "SchoolID", "left"))
filled = s.filter(F.col("ProvinceCode").isNull() & F.col("SchoolProvince").isNotNull())
log_issues(filled, "sessions", "Blank province on school session", "Corrected (from school)", "SessionID",
           F.concat(F.lit("-> "), F.col("SchoolProvince")))
s = (s.withColumn("ProvinceCode", F.coalesce("ProvinceCode", "SchoolProvince")).drop("SchoolProvince"))
still_blank = s.filter(F.col("ProvinceCode").isNull())
log_issues(still_blank, "sessions", "Province missing", "Flagged", "SessionID", F.col("ProgrammeID"))
print(f"Sessions raw {raw_count} | resubmissions removed {resub.count()} | provinces filled {filled.count()} "
      f"| still blank {still_blank.count()} | clean {s.count()}")

## 5. Attendance: quarantine unknown participants
Each row says "this participant attended this session". We check both sides exist:
the participant must be in the register, and the session must be a real session.

In [ ]:
a = read_csv(f"{FIELD}/attendance/*.csv").select(F.trim("SessionID").alias("SessionID"),
                                                  F.trim("ParticipantID").alias("ParticipantID"))
raw_a = a.count()
a = a.dropDuplicates(["SessionID", "ParticipantID"])
dupe_a = raw_a - a.count()

orphan_p = a.join(p.select("ParticipantID"), "ParticipantID", "left_anti")
log_issues(orphan_p, "attendance", "Participant not in register", "Quarantined", "ParticipantID", F.col("SessionID"))
orphan_s = a.join(s.select("SessionID"), "SessionID", "left_anti")
log_issues(orphan_s, "attendance", "Session not found", "Quarantined", "SessionID", F.col("ParticipantID"))

a = a.join(p.select("ParticipantID"), "ParticipantID", "left_semi").join(s.select("SessionID"), "SessionID", "left_semi")
print(f"Attendance raw {raw_a} | exact duplicates {dupe_a} | unknown participant {orphan_p.count()} "
      f"| unknown session {orphan_s.count()} | clean {a.count()}")

## 6. School attendance by term
Rates from provincial education offices. Just typing, plus a sanity check that every rate is between 0 and 1.

In [ ]:
sa = (read_csv(f"{FIELD}/school_attendance_by_term.csv")
      .withColumn("Year", F.col("Year").cast("int")).withColumn("Term", F.col("Term").cast("int"))
      .withColumn("TermStart", F.to_date("TermStart")).withColumn("TermEnd", F.to_date("TermEnd"))
      .withColumn("Enrolled", F.col("Enrolled").cast("int"))
      .withColumn("AvgAttendanceRate", F.col("AvgAttendanceRate").cast("double")))
bad_rate = sa.filter((F.col("AvgAttendanceRate") < 0) | (F.col("AvgAttendanceRate") > 1))
log_issues(bad_rate, "school_attendance", "Rate outside 0 to 1", "Flagged", "SchoolID", F.col("AvgAttendanceRate"))
print("School-terms:", sa.count(), "| bad rates:", bad_rate.count())

## 7. Write silver tables and the DQ log

In [ ]:
def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{name}")

save(provinces, "provinces")
save(schools, "schools")
save(p, "participants")
save(s, "sessions")
save(a, "attendance")
save(sa, "school_attendance")

dq = reduce(lambda x, y: x.unionByName(y), dq_frames).withColumn("LoggedAtUtc", F.current_timestamp())
dq.write.mode("append").saveAsTable(f"{SILVER}.dq_log")
display(dq.groupBy("SourceTable", "Rule", "Action").count().orderBy("SourceTable", "Rule"))

## 8. Checks
Expected (answer key): 6,400 in the register, of whom 6,398 attended at least once (reach); 2,003 sessions; 73,775 attendances; 30 resubmissions removed;
40 provinces filled; 22 attendance rows quarantined; 384 gender values corrected.
Reach is **distinct participants who attended**, not attendances.

In [ ]:
att = spark.read.table(f"{SILVER}.attendance")
ses = spark.read.table(f"{SILVER}.sessions")
par = spark.read.table(f"{SILVER}.participants")
print("Sessions:", ses.count(), "| Attendances:", att.count(), "| Participants:", par.count())

reach = (att.join(ses.select("SessionID", "ProgrammeID"), "SessionID")
            .join(par.select("ParticipantID", "Gender"), "ParticipantID")
            .groupBy("ProgrammeID")
            .agg(F.countDistinct("ParticipantID").alias("Participants"), F.count("*").alias("Attendances"),
                 F.round(F.countDistinct(F.when(F.col("Gender") == "F", F.col("ParticipantID")))
                         / F.countDistinct("ParticipantID"), 3).alias("FemaleShare"))
            .orderBy("ProgrammeID"))
display(reach)